# Experiment 3 · Run C: a student of the uniform-average teacher

Does learned attention matter? **C1** trains a second student identical to Experiment 2's step 3 except the target: the uniform average of each point's DINOv2 observations (checked against the pilot's lifted feature before training; the only configuration difference is checked too). **C2** evaluates it exactly like Experiment 2's step 4 (f(p) and the head features) and compares it with the attention student by paired bootstrap. Then the report over all three runs. Outputs: /kaggle/working/Experiment3_DistillationDiagnostics/. Import this notebook into the pinned `pilot_03_probes` notebook (File → Import Notebook). Set RUN below: "smoke" first, then "full". Training checkpoints every 30 min and resumes from an attached earlier output of the same mode.

**Before running**
- [ ] **Accelerator:** GPU T4 x2
- [ ] **Internet:** on
- [ ] **Secrets** (Add-ons → Secrets): `GITHUB_TOKEN` attached
- [ ] **Environment:** pin to the same image as the wheelhouse (Settings → Environment preference); install_env.py refuses a mismatched image
- [ ] **Input:** private dataset `appearance-pt-wheels` (output of m1_01)
- [ ] **Inputs:** private datasets `scannetpp-core-val`, `scannetpp-core-train-a` and `scannetpp-core-train-b`
- [ ] **Input:** the `pilot_04_eval` notebook output (Add Input → Your Work → Notebooks → pilot_04_eval): its `pilot_concat/step5/confmats.npz` and `results.json` give the two Sonata baseline rows
- [ ] **Input:** the output of the full Experiment 2 run (the `pilot_02_concerto_lift` notebook's latest version, where `exp2_cross_attention` was imported): its student, teacher, observations and evaluation are reused
- [ ] **Input:** the output of the Experiment 3 A+B notebook (`exp3_ab_diagnostics`, same RUN mode): Run A's confusion matrices give the A2/A3 comparisons, and the report then covers all three runs
- [ ] **Input (only to resume an interrupted run):** this notebook's own earlier output of the same mode

**Run:** Save Version → Save & Run All (Commit). Smoke: about 20–30 min. Full: about 3.3 h (training 2.6 h as in Experiment 2, evaluation about 40 min); needs the user's go-ahead (> 2 h).

**Paste back:** the SUMMARY blocks of C1 and C2 (logs: /kaggle/working/logs/exp3_*.log); the report is /kaggle/working/Experiment3_DistillationDiagnostics/reports/distillation_diagnostics_report.pdf.

*Generated by `scripts/make_notebooks.py`; edit the scripts, not this notebook.*

In [ ]:
# ---- run configuration: the only cell to edit ----
REPO = "eylulpelinkilic/appearance-pt"  # your private GitHub repo (owner/name)
REF = "main"                  # branch, tag or commit SHA; the resolved commit is logged

In [ ]:
"""Kaggle bootstrap: fetch this (private) repo at a fixed ref, then run entry points with logs.

Embedded verbatim in every notebook by scripts/make_notebooks.py; edit here, then regenerate.
The GitHub token comes from Kaggle Secrets and reaches git only through environment variables
(never argv, files or output), so it cannot leak into logs or tracebacks; git's error messages
are shown (a notebook would otherwise hide them) with the token redacted.
"""

import base64
import os
import pathlib
import shutil
import subprocess
import sys
import time

_GIT_DEBUG_VARS = ("GIT_TRACE", "GIT_TRACE_CURL", "GIT_CURL_VERBOSE", "GIT_TRACE_PACKET")


def _git(args, cwd=None, env=None, capture=False, redact=()):
    # stderr is captured: a notebook cell never shows a subprocess's stderr, only the exit code
    r = subprocess.run(["git", *args], cwd=cwd, env=env, text=True, capture_output=True)
    if r.returncode != 0:
        err = (r.stderr or "").strip()
        for s in redact:
            err = err.replace(s, "***")
        raise RuntimeError(f"git {args[0]} failed (exit {r.returncode}):\n{err}")
    return r.stdout.strip() if capture else None


def clone_repo(repo, ref, dest="/tmp/appearance-pt", get_secret=None):
    """Fetch `repo` ("owner/name") at `ref` (branch, tag or commit) into `dest`.

    Submodules (public) are fetched at their pinned commits. Returns (dest, commit).
    """
    if repo.startswith("OWNER/"):
        raise ValueError('set REPO in the config cell to your GitHub repo ("owner/name")')
    if get_secret is None:
        from kaggle_secrets import UserSecretsClient

        get_secret = UserSecretsClient().get_secret
    token = get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {k: v for k, v in os.environ.items() if k not in _GIT_DEBUG_VARS}
    env.update(
        GIT_TERMINAL_PROMPT="0",
        GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
        GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}",
    )
    hide = (token, auth)
    del token, auth
    dest = pathlib.Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    _git(["init", "-q", str(dest)])
    _git(["remote", "add", "origin", f"https://github.com/{repo}.git"], cwd=dest)
    try:
        _git(["fetch", "-q", "--depth", "1", "origin", ref], cwd=dest, env=env, redact=hide)
    except RuntimeError as e:
        raise RuntimeError(
            f"{e}\n\nCould not fetch {repo} @ {ref}. Check: REPO/REF in the config cell; Internet on; "
            "GITHUB_TOKEN attached to this notebook and allowed to read this repo "
            "(fine-grained token: repo selected under 'Repository access', 'Contents: Read-only', not expired)."
        ) from None
    finally:
        del env, hide
    _git(["checkout", "-q", "FETCH_HEAD"], cwd=dest)
    _git(["submodule", "update", "--init", "--depth", "1", "-q"], cwd=dest)
    commit = _git(["rev-parse", "HEAD"], cwd=dest, capture=True)
    print(f"repo {repo} @ {ref} -> {commit}\n{_git(['submodule', 'status'], cwd=dest, capture=True)}")
    return dest, commit


def run_logged(cmd, log_path, cwd, extra_env=None):
    """Run `cmd`, streaming output to the notebook and appending it to `log_path`.

    Raises if the command fails, so a failed step stops "Save & Run All" visibly.
    """
    log_path = pathlib.Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", **(extra_env or {})}
    t0 = time.time()
    with open(log_path, "a") as log:
        log.write(f"\n### {time.strftime('%Y-%m-%d %H:%M:%S')} $ {' '.join(map(str, cmd))}\n")
        with subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
            for line in p.stdout:
                sys.stdout.write(line)
                log.write(line)
            rc = p.wait()
    print(f"\n[exit code {rc} after {(time.time() - t0) / 60:.1f} min; log: {log_path}]")
    if rc != 0:
        raise RuntimeError(f"step failed with exit code {rc}; see {log_path}")


CODE_DIR, COMMIT = clone_repo(REPO, REF)
PYTHONPATH = {"PYTHONPATH": f"{CODE_DIR}/src"}

In [ ]:
# Experiment 3 imports the encoder packages (pinned submodules), our src/, and Experiments 1 and 2's modules (unchanged)
EXP3_PYTHONPATH = {"PYTHONPATH": ":".join(f"{CODE_DIR}/{p}" for p in ("src", "third_party/utonia", "third_party/sonata",
                                                                     "third_party/concerto", "experiments/Experiment1_Concatenation",
                                                                     "experiments/Experiment2_CrossAttention"))}

In [ ]:
# "smoke" first (a few scenes through every step, with all checks); then "full"
RUN = "smoke"

In [ ]:
# the pilot's saved step-5 outputs (pilot_concat/step5/ of pilot_04_eval): the two Sonata baseline rows are reused from them
# (the walk skips the ScanNet++ scene folders: an rglob over them takes minutes)
import os
import pathlib
import numpy as np
found = []
for d, sub, files in os.walk("/kaggle/input"):
    sub[:] = sorted(s for s in sub if s not in ("scenes", "wheelhouse"))
    if "confmats.npz" in files and "results.json" in files and "S#0|top1" in np.load(os.path.join(d, "confmats.npz")).files:
        found.append(pathlib.Path(d))
assert found, "attach the pilot_04_eval notebook output (pilot_concat/step5/: confmats.npz and results.json)"
PILOT = str(found[0])
print(f"pilot outputs: {PILOT}")

In [ ]:
run_logged([sys.executable, "-u", "scripts/setup/install_env.py"],
           "/kaggle/working/logs/exp3_install.log", cwd=CODE_DIR, extra_env=PYTHONPATH)

In [ ]:
# the report builder needs reportlab (not in the Kaggle image or the wheelhouse); matplotlib is in the image
run_logged([sys.executable, "-m", "pip", "install", "--quiet", "reportlab==4.2.5"], "/kaggle/working/logs/exp3_pip.log", cwd=CODE_DIR)

In [ ]:
run_logged([sys.executable, "-u", "experiments/Experiment3_DistillationDiagnostics/stepC1_uniform_student.py", "--mode", RUN],
           "/kaggle/working/logs/exp3_runC1.log", cwd=CODE_DIR, extra_env=EXP3_PYTHONPATH)

In [ ]:
try:
    run_logged([sys.executable, "-u", "experiments/Experiment3_DistillationDiagnostics/stepC2_evaluate.py", "--mode", RUN, "--pilot", PILOT],
               "/kaggle/working/logs/exp3_runC2.log", cwd=CODE_DIR, extra_env=EXP3_PYTHONPATH)
except RuntimeError as e:
    print(f"!! {e}; going on with the next step")

In [ ]:
run_logged([sys.executable, "-u", "experiments/Experiment3_DistillationDiagnostics/step_report.py", "--mode", RUN],
           "/kaggle/working/logs/exp3_report.log", cwd=CODE_DIR, extra_env=EXP3_PYTHONPATH)

In [ ]:
# what this run produced (download /kaggle/working/Experiment3_DistillationDiagnostics from the Output tab)
for p in sorted(pathlib.Path("/kaggle/working/Experiment3_DistillationDiagnostics/reports").glob("*")):
    print(f"{p}  {p.stat().st_size / 1e6:.2f} MB")